# 06 — Terrain Derivation

Pre-event optical imagery is a weak forecasting signal: a stable forested
slope and one about to fail look nearly identical from above. Terrain is
what carries the prediction.

Strategy:

| Decision | Choice | Why |
|---|---|---|
| DEM source | Copernicus DEM GLO-30 | Free, global, 30 m, no registration. Best quality open DEM for Sri Lanka. |
| Download method | One regional mosaic, cached | All patches fall in ~4 DEM tiles. Per-patch fetching would repeat the same download thousands of times. |
| Derived layers | slope, hillshade, profile curvature | Steepness, shaded relief (the "contour map" view), and concave hollows where water and debris concentrate. |
| Contour render | generated, not model input | Contours are decoded elevation — lossy. Kept for the ablation and the UI. |
| Alignment | same UTM grid, same 64 px, same bounds | Every terrain pixel matches its optical pixel exactly. |

Output: data/terrain/patches/<id>.tif (3 bands) + contour PNGs

> Copernicus DEM GLO-30 via Microsoft Planetary Computer.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer as pc
import pystac_client
import rasterio
from odc.geo.geobox import GeoBox
from odc.stac import load as odc_load
from rasterio.enums import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.windows import from_bounds as window_from_bounds
from tqdm.auto import tqdm

PROJECT_DIR  = Path("..").resolve()
COORD_CSV    = PROJECT_DIR / "data" / "coordinates" / "all_coordinates.csv"
TERRAIN_DIR  = PROJECT_DIR / "data" / "terrain"
PATCH_DIR    = TERRAIN_DIR / "patches"
CONTOUR_DIR  = TERRAIN_DIR / "contours"
FIG_DIR      = PROJECT_DIR / "outputs" / "figures"
DEM_MOSAIC   = TERRAIN_DIR / "dem_mosaic.tif"

for d in (PATCH_DIR, CONTOUR_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

PATCH_SIZE_M = 640          # must match the acquisition notebook
PATCH_PIXELS = 64
DST_CRS      = "EPSG:32644"
DEM_RES_M    = 10           # mosaic resampled to match Sentinel-2

HILLSHADE_AZIMUTH  = 315.0
HILLSHADE_ALTITUDE = 45.0
CONTOUR_INTERVAL_M = 20

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

print(f"patch {PATCH_SIZE_M} m / {PATCH_PIXELS} px @ "
      f"{PATCH_SIZE_M/PATCH_PIXELS:.0f} m/px")

In [ ]:
coords = pd.read_csv(COORD_CSV)
print(f"{len(coords)} locations "
      f"({(coords.label==1).sum()} pos, {(coords.label==0).sum()} neg)")

gdf = gpd.GeoDataFrame(coords, geometry=gpd.points_from_xy(
    coords.longitude, coords.latitude), crs="EPSG:4326").to_crs(DST_CRS)

half = PATCH_SIZE_M / 2
gdf["minx"] = gdf.geometry.x - half
gdf["maxx"] = gdf.geometry.x + half
gdf["miny"] = gdf.geometry.y - half
gdf["maxy"] = gdf.geometry.y + half

locations = gdf[["landslide_id", "label",
                 "minx", "miny", "maxx", "maxy"]].to_dict("records")
print(locations[0])

In [ ]:
PAD_M = 3000

if DEM_MOSAIC.exists():
    with rasterio.open(DEM_MOSAIC) as s:
        print(f"cached: {s.width} x {s.height} px "
              f"({DEM_MOSAIC.stat().st_size/1e6:.0f} MB)")
else:
    bounds = (gdf.minx.min()-PAD_M, gdf.miny.min()-PAD_M,
              gdf.maxx.max()+PAD_M, gdf.maxy.max()+PAD_M)
    gbox = GeoBox.from_bbox(bounds, crs=DST_CRS, resolution=DEM_RES_M)
    print(f"fetching DEM {gbox.shape} @ {DEM_RES_M} m ...")

    cat = pystac_client.Client.open(STAC_URL, modifier=pc.sign_inplace)
    bb = gbox.geographic_extent.boundingbox
    items = list(cat.search(collections=["cop-dem-glo-30"],
                 bbox=[bb.left, bb.bottom, bb.right, bb.top]).items())
    print(f"  {len(items)} DEM tiles")
    assert items, "no DEM coverage"

    dem = odc_load(items, bands=["data"], geobox=gbox,
                   resampling="bilinear", chunks=None,
                   patch_url=pc.sign)["data"].values
    if dem.ndim == 3:
        dem = np.nanmax(dem, axis=0)

    with rasterio.open(DEM_MOSAIC, "w", driver="GTiff",
                       height=dem.shape[0], width=dem.shape[1], count=1,
                       dtype="float32", crs=DST_CRS,
                       transform=gbox.transform, compress="deflate") as dst:
        dst.write(dem.astype("float32"), 1)

    print(f"  saved -> {DEM_MOSAIC} "
          f"({DEM_MOSAIC.stat().st_size/1e6:.0f} MB)")

In [ ]:
def read_dem_window(bounds, px=PATCH_PIXELS):
    """Cut a patch-sized DEM window from the cached mosaic."""
    with rasterio.open(DEM_MOSAIC) as src:
        win = window_from_bounds(*bounds, transform=src.transform)
        z = src.read(1, window=win, out_shape=(px, px),
                     resampling=Resampling.bilinear,
                     boundless=True, fill_value=np.nan)
    return z.astype(np.float32)


def derive_terrain(z, bounds, px=PATCH_PIXELS):
    """
    Returns (3, px, px) float32: slope (deg), hillshade (0-255), curvature.
    """
    res = (bounds[2] - bounds[0]) / px
    z = np.nan_to_num(z, nan=float(np.nanmedian(z)))

    dy, dx = np.gradient(z, res)
    slope_rad = np.arctan(np.hypot(dx, dy))
    slope_deg = np.degrees(slope_rad)

    # hillshade — shaded relief, the topographic-map view
    aspect = np.arctan2(dy, -dx)
    zen = np.radians(90.0 - HILLSHADE_ALTITUDE)
    azi = np.radians(360.0 - HILLSHADE_AZIMUTH + 90.0)
    hs = (np.cos(zen) * np.cos(slope_rad)
          + np.sin(zen) * np.sin(slope_rad) * np.cos(azi - aspect))
    hs = np.clip(hs, 0, 1) * 255.0

    # profile curvature — negative = concave hollow (water/debris concentrates)
    dyy, _ = np.gradient(dy, res)
    _, dxx = np.gradient(dx, res)
    curv = (dxx + dyy) * 100.0     # scaled for numerical range

    return np.stack([slope_deg, hs, curv]).astype(np.float32)


def save_terrain(pid, arr, bounds):
    tf = transform_from_bounds(*bounds, arr.shape[2], arr.shape[1])
    path = PATCH_DIR / f"{pid}.tif"
    with rasterio.open(path, "w", driver="GTiff",
                       height=arr.shape[1], width=arr.shape[2], count=3,
                       dtype="float32", crs=DST_CRS, transform=tf,
                       compress="deflate") as dst:
        dst.write(arr)
        dst.descriptions = ("slope_deg", "hillshade", "curvature")
    return str(path)

In [ ]:
loc = locations[0]
b = (loc["minx"], loc["miny"], loc["maxx"], loc["maxy"])
z = read_dem_window(b)
t = derive_terrain(z, b)

fig, ax = plt.subplots(1, 4, figsize=(17, 4.2))
im0 = ax[0].imshow(z, cmap="terrain");     ax[0].set_title("Elevation (m)")
ax[1].imshow(t[0], cmap="magma");          ax[1].set_title("Slope (deg)")
ax[2].imshow(t[1], cmap="gray");           ax[2].set_title("Hillshade")
ax[3].imshow(t[2], cmap="RdBu_r",
             vmin=-np.abs(t[2]).max(), vmax=np.abs(t[2]).max())
ax[3].set_title("Curvature")
for a in ax: a.axis("off")
plt.colorbar(im0, ax=ax[0], fraction=.046)
plt.tight_layout(); plt.show()

print(f"elevation {z.min():.0f}-{z.max():.0f} m  |  "
      f"slope {t[0].min():.1f}-{t[0].max():.1f} deg  "
      f"(mean {t[0].mean():.1f})")

## Checkpoint

- Elevation range is plausible for the central highlands (roughly 100-2500 m)
- Slope shows structure, not a flat field
- Hillshade shows recognisable ridges and valleys
- Mean slope on a landslide patch should be well above 15 degrees

If elevation is constant or zero, the DEM window is outside the mosaic —
check the bounds and CRS before running the batch.

def save_contour(pid, z, px=256, interval=CONTOUR_INTERVAL_M):
    """Rendered contour map — for the ablation experiment and the UI only."""
    fig, ax = plt.subplots(figsize=(px/100, px/100), dpi=100)
    ax.set_facecolor("#f7f2d3")
    lo = np.floor(z.min()/interval)*interval
    levels = np.arange(lo, z.max()+interval, interval)
    if len(levels) > 1:
        ax.contour(z, levels=levels, colors="#6b5a20", linewidths=0.5)
        ax.contour(z, levels=levels[::5], colors="#3a2e08", linewidths=1.1)
    ax.set_xlim(0, z.shape[1]); ax.set_ylim(z.shape[0], 0)
    ax.axis("off"); plt.subplots_adjust(0, 0, 1, 1)
    path = CONTOUR_DIR / f"{pid}.png"
    plt.savefig(path, dpi=100); plt.close(fig)
    return str(path)

# preview
_ = save_contour(loc["landslide_id"], z)
plt.figure(figsize=(4, 4))
plt.imshow(plt.imread(CONTOUR_DIR / f"{loc['landslide_id']}.png"))
plt.axis("off"); plt.title("Contour render"); plt.show()

In [ ]:
MAKE_CONTOURS = True     # set False to skip; roughly doubles runtime

records = []
for loc in tqdm(locations):
    pid = loc["landslide_id"]
    b = (loc["minx"], loc["miny"], loc["maxx"], loc["maxy"])
    try:
        z = read_dem_window(b)
        if not np.isfinite(z).any():
            records.append({"landslide_id": pid, "label": loc["label"],
                            "status": "no_dem"})
            continue
        t = derive_terrain(z, b)
        tif = save_terrain(pid, t, b)
        png = save_contour(pid, z) if MAKE_CONTOURS else ""
        records.append({
            "landslide_id": pid, "label": loc["label"], "status": "success",
            "terrain_tif": tif, "contour_png": png,
            "elev_mean": float(z.mean()), "elev_min": float(z.min()),
            "elev_max": float(z.max()), "relief_m": float(z.max()-z.min()),
            "slope_mean": float(t[0].mean()), "slope_max": float(t[0].max()),
            "slope_p90": float(np.percentile(t[0], 90)),
            "curv_mean": float(t[2].mean()), "curv_std": float(t[2].std()),
        })
    except Exception as exc:
        records.append({"landslide_id": pid, "label": loc["label"],
                        "status": "error", "error": repr(exc)})

terrain = pd.DataFrame(records)
terrain.to_csv(PROJECT_DIR / "data" / "metadata" / "terrain_features.csv",
               index=False)
print(terrain.status.value_counts())

In [ ]:
ok = terrain.query("status == 'success'")
pos = ok[ok.label == 1]; neg = ok[ok.label == 0]

print(f"{'feature':<12} {'positive':>12} {'negative':>12} {'diff':>8}")
print("-" * 48)
for f in ["slope_mean", "slope_p90", "slope_max",
          "relief_m", "elev_mean", "curv_std"]:
    a, b_ = pos[f].mean(), neg[f].mean()
    print(f"{f:<12} {a:>12.2f} {b_:>12.2f} {a-b_:>8.2f}")

from scipy.stats import mannwhitneyu
u, p_val = mannwhitneyu(pos.slope_mean, neg.slope_mean)
print(f"\nmean slope, Mann-Whitney U p = {p_val:.2e}")

In [ ]:
feats = ["slope_mean", "slope_p90", "relief_m", "curv_std"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, f in zip(axes, feats):
    ax.hist(neg[f], bins=40, alpha=.6, label="negative", color="steelblue",
            density=True)
    ax.hist(pos[f], bins=40, alpha=.6, label="positive", color="firebrick",
            density=True)
    ax.set(title=f, ylabel="density")
    ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "06_terrain_separation.png", dpi=130, bbox_inches="tight")
plt.show()

In [ ]:
ids = list(pos.landslide_id.head(4)) + list(neg.landslide_id.head(4))
fig, axes = plt.subplots(3, 8, figsize=(20, 7.5))
for j, pid in enumerate(ids):
    with rasterio.open(PATCH_DIR / f"{pid}.tif") as s:
        a = s.read()
    lab = "POS" if pid in set(pos.landslide_id) else "NEG"
    axes[0, j].imshow(a[0], cmap="magma")
    axes[0, j].set_title(f"{lab}\n{pid}", fontsize=7)
    axes[1, j].imshow(a[1], cmap="gray")
    axes[2, j].imshow(a[2], cmap="RdBu_r")
for a_ in axes.ravel(): a_.axis("off")
for i, lbl in enumerate(["Slope", "Hillshade", "Curvature"]):
    axes[i, 0].set_ylabel(lbl)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_terrain_grid.png", dpi=130, bbox_inches="tight")
plt.show()

## Checklist

- [ ] DEM mosaic cached; no repeated downloads
- [ ] All patches processed, few or no failures
- [ ] Positive patches show higher mean slope than negatives
- [ ] Mann-Whitney p < 0.01 (terrain carries real signal)
- [ ] terrain_features.csv written

Commit: "Add DEM mosaic, terrain derivation and contour rendering"

Next: 07_build_dataset.ipynb — spatial split, 8-channel stacking, normalisation.